In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import RandomizedSearchCV
import mlflow
import pickle
from pathlib import Path

# ==========================================
# 1. SETUP PATH & MLFLOW
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("07_XGBoost_Hyperparameter_Tuning")

print("⏳ 1. Memuat Data Hasil Seleksi Fitur MFO...")
train_df = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")

# Fitur yang terpilih dari output MFO Anda
selected_features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 
                     'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 
                     'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 
                     'religion', 'orientation', 'race', 'voted', 'married']

target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

X_train = train_df[selected_features]
Y_train = train_df[target_cols]

with mlflow.start_run(run_name="XGBoost_MultiLabel_Tuning"):
    # ==========================================
    # 2. DEFINISI RUANG HYPERPARAMETER
    # ==========================================
    print("⏳ 2. Mencari Parameter Terbaik (Hyperparameter Tuning)...")
    
    param_grid = {
        'estimator__n_estimators': [100, 200, 300],
        'estimator__max_depth': [3, 5, 7, 9],
        'estimator__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'estimator__subsample': [0.7, 0.8, 0.9],
        'estimator__colsample_bytree': [0.7, 0.8, 0.9],
        'estimator__gamma': [0, 0.1, 0.2]
    }

    base_xgb = xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
    multi_target_xgb = MultiOutputClassifier(base_xgb)

    # Menggunakan RandomizedSearch untuk efisiensi waktu
    random_search = RandomizedSearchCV(
        multi_target_xgb, 
        param_distributions=param_grid, 
        n_iter=10, 
        cv=3, 
        scoring='f1_macro', 
        verbose=1, 
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train, Y_train)

    best_params = random_search.best_params_
    best_score = random_search.best_score_

    print(f"\n✅ Parameter Terbaik Ditemukan: {best_params}")
    print(f"✅ Estimasi F1-Score (Internal CV): {best_score:.4f}")

    # ==========================================
    # 3. PELATIHAN MODEL FINAL
    # ==========================================
    print("⏳ 3. Melatih Model Final dengan Parameter Terbaik...")
    final_model = random_search.best_estimator_
    final_model.fit(X_train, Y_train)

    # ==========================================
    # 4. PENYIMPANAN MODEL
    # ==========================================
    model_dir = root_path / "models"
    model_dir.mkdir(parents=True, exist_ok=True)
    
    with open(model_dir / "multilabel_xgboost_mfo.pkl", "wb") as f:
        pickle.dump(final_model, f)

    # Logging ke MLflow
    for param, value in best_params.items():
        mlflow.log_param(param, value)
    mlflow.log_metric("best_cv_f1_score", best_score)
    
    print(f"\n🎉 Model Final Berhasil Disimpan di {model_dir}")

⏳ 1. Memuat Data Hasil Seleksi Fitur MFO...
⏳ 2. Mencari Parameter Terbaik (Hyperparameter Tuning)...
Fitting 3 folds for each of 10 candidates, totalling 30 fits


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [14:08:26] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [14:08:27] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [14:08:27] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are no


✅ Parameter Terbaik Ditemukan: {'estimator__subsample': 0.7, 'estimator__n_estimators': 300, 'estimator__max_depth': 3, 'estimator__learning_rate': 0.05, 'estimator__gamma': 0.2, 'estimator__colsample_bytree': 0.7}
✅ Estimasi F1-Score (Internal CV): 0.4874
⏳ 3. Melatih Model Final dengan Parameter Terbaik...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [14:08:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [14:08:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [14:08:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are no


🎉 Model Final Berhasil Disimpan di d:\Amikom\Semester 6\Project Data Mining\Project\models
